# 08. Loop checks, Denmark-wide
## Project: Bicycle node network loop analysis

This notebook explores the loops created in notebook 02 by number of nodes and length limitations, for the whole country of Denmark.  
<font color='red'>Select `denmark` as the `study_area` in the `config.yml`.</font>

Created: 2026-03-09  
Last modified: 2026-03-09

## Parameters

In [ ]:
%run -i setup_parameters.py
load_data = True  # Set to False if data are huge and have already been loaded
debug = True  # Set to True for extra plots and verbosity

## Functions

In [ ]:
%run -i functions.py

## Load data

This can take several minutes.

In [ ]:
if load_data:
    if LOOP_LENGTH_BOUND:
        llb_string = "_maxlength" + str(LOOP_LENGTH_BOUND)
    else:
        llb_string = ""

    with open(
        PATH["data_out"]
        + "loopcensus_"
        + str(LOOP_NUMNODE_BOUND)
        + llb_string
        + ".pkl",
        "rb",
    ) as f:
        allloops = pickle.load(f)
        alllooplengths = pickle.load(f)
        allloopnumnodes = pickle.load(f)
        allloopmaxslopes = pickle.load(f)
        Gnx = pickle.load(f)
        LOOP_NUMNODE_BOUND = pickle.load(f)
        nodes_id = pickle.load(f)
        nodes_coords = pickle.load(f)
        numloops = pickle.load(f)
        faceloops = pickle.load(f)

## All loops

This can take a few minutes.

In [ ]:
fig = plt.figure(figsize=(8, 3))
axes1 = fig.add_axes([0.1, 0.1, 0.35, 0.8])
axes2 = fig.add_axes([0.55, 0.1, 0.35, 0.8])

axes1.hist(alllooplengths, bins=range(0, 40001, 1000), density=True)
if MPERUNIT == 1000:
    axes1.set_xlabel("Length [km]")
elif MPERUNIT == 1:
    axes1.set_xlabel("Length [m]")
else:
    axes1.set_xlabel("Length")
axes1.set_ylabel("Probability")
axes1.set_title("Loop lengths")
axes1.set_xlim(left=0)

axes2.hist(allloopnumnodes, density=True, bins=list(range(LOOP_NUMNODE_BOUND + 2)))
axes2.set_xlabel("Nodes")
axes2.set_title("Nodes per loop")
axes2.set_xlim([0, LOOP_NUMNODE_BOUND + 1.5])

plt.text(LOOP_NUMNODE_BOUND / 20, 0.03, "Bound: " + str(LOOP_NUMNODE_BOUND))
plt.text(LOOP_NUMNODE_BOUND / 20, 0.04, "Loops: " + str(numloops));

## Loops for adults

In [ ]:
loop_numnode_bounds = list(range(5, 31, 5))
fig, axes = plt.subplots(
    nrows=2, ncols=3, figsize=(8, 4)
)  # nrows specifies the number of rows you want, ncols how many figures in each row
axs = axes.flatten()

i = 0
for axes1, bd in zip(axs, loop_numnode_bounds):
    indmaxnodes = np.where(allloopnumnodes == bd)[0]
    if len(indmaxnodes):
        axes1.hist(
            alllooplengths[indmaxnodes], bins=range(0, 40001, 1000), density=True
        )
        if MPERUNIT == 1000:
            axes1.set_xlabel("Length [km]")
        elif MPERUNIT == 1:
            axes1.set_xlabel("Length [m]")
        else:
            axes1.set_xlabel("Length")
        if not i % 3:
            axes1.set_ylabel("Probability")
        else:
            axes1.set_yticklabels({})
        if i < 3:
            axes1.set_xticklabels({})
            axes1.set_xlabel("")
        axes1.set_title("Loop lengths for " + str(bd) + " nodes")

        axes1.set_xlim(left=0)
        axes1.set_xlim(right=LOOP_LENGTH_BOUND * 1.05)
        axes1.set_ylim([0, 0.00028])

        axes1.text(
            1400, 0.00025, "Loops: " + "{:,}".format(len(alllooplengths[indmaxnodes]))
        )
    else:
        print("No loops found at number of nodes bound " + str(LOOP_NUMNODE_BOUND))
    i += 1

fig.set_tight_layout(True)
fig.savefig(PATH["plot"] + "loopcapadult" + ".pdf");

## Loops for families

In [ ]:
loop_numnode_bounds = list(range(5, 31, 5))
fig, axes = plt.subplots(
    nrows=2, ncols=3, figsize=(8, 4)
)  # nrows specifies the number of rows you want, ncols how many figures in each row
axs = axes.flatten()

i = 0
for axes1, bd in zip(axs, loop_numnode_bounds):
    indmaxnodes = np.where(allloopnumnodes == bd)[0]
    if len(indmaxnodes):
        axes1.hist(
            alllooplengths[indmaxnodes], bins=range(0, 40001, 1000), density=False
        )
        h = np.histogram(
            alllooplengths[indmaxnodes], bins=range(0, 40001, 1000), density=False
        )
        if MPERUNIT == 1000:
            axes1.set_xlabel("Length [km]")
        elif MPERUNIT == 1:
            axes1.set_xlabel("Length [m]")
        else:
            axes1.set_xlabel("Length")
        if not i % 3:
            axes1.set_ylabel("Frequency")
        else:
            axes1.set_yticklabels({})
        if i < 3:
            axes1.set_xticklabels({})
            axes1.set_xlabel("")
        axes1.set_title("Loop lengths for " + str(bd) + " nodes")

        axes1.set_xlim(left=0)
        axes1.set_xlim(right=20000)
        axes1.set_ylim([0, 4000])

        axes1.text(400, 3600, "Loops below 20 km: " + "{:,}".format(sum(h[0][0:20])))
    else:
        print("No loops found at number of nodes bound " + str(LOOP_NUMNODE_BOUND))
    i += 1

fig.set_tight_layout(True)
fig.savefig(PATH["plot"] + "loopcapfamily" + ".pdf");